In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.plotting import (
    MovieRenderConfig,
    inspect_tiff_layout,
    render_glutamate_df_movie,
    load_slap2_movie_from_tiffs
)
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary

In [ ]:
%matplotlib notebook

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def display_meanim(exp, dmd=1, image_type="meanIM", channel=0, robust=(1, 99)):
    im = exp.get_summary_image(dmd=dmd, image_type=image_type)
    if im.shape[1]>im.shape[0]:
        im = im.T

    if im.ndim == 3:
        if channel >= im.shape[1]:
            raise ValueError(f"Requested channel {channel}, but image has shape {im.shape}")
        im = im[:, channel, :]
    elif im.ndim != 2:
        raise ValueError(f"Unexpected meanIM shape: {im.shape}")

    im = np.nan_to_num(im, nan=0.0)
    vmin, vmax = np.percentile(im, robust)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(im.T, cmap="viridis", vmin=vmin, vmax=vmax, origin="lower")
    ax.set_title(f"DMD{dmd} {image_type}, channel {channel}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    return fig, ax

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SESSION_ID = None  # e.g. "834788_2026-03-04_08-43-07"; leave as None to use SUBJECT_ID + SESSION_INDEX
SUBJECT_ID = 803496
SESSION_INDEX = 4  # used only when SESSION_ID is None

FS_HZ = 200.0
DMD_FOR_CALCIUM = 2
ROI_INDICES_TO_PLOT = [0,1,2]          # whole-session calcium traces
MAX_SESSION_MINUTES = None         # e.g. 10.0 to truncate late-session data
MOTION_CORRECT = True
TRACE_TYPE = "Fsvd"
CALCIUM_TRACE_KEY = "dff"         # one of: dff, ca_mc, ca_unmixed, ca_clean

DMD_FOR_MEANIM = 1
MEANIM_IMAGE_TYPE = "meanIM"
MEANIM_CHANNEL = 0

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

if SESSION_ID is not None:
    session_row = registry.get_session_row(SESSION_ID)
else:
    subject_sessions = registry.sessions(subject_ids=[SUBJECT_ID]).reset_index(drop=True)
    if len(subject_sessions) == 0:
        raise ValueError(f"No sessions found for subject {SUBJECT_ID}")
    session_row = subject_sessions.iloc[SESSION_INDEX]

asset = registry.resolve_assets(session_row)
asset.ensure_dirs()
exp = GlutamateSummary(asset.summary_mat, keep_open=True)

print(f"session_id: {asset.session_id}")
print(f"subject_id: {asset.subject_id}")
print(f"summary_mat: {asset.summary_mat}")
print(f"bonsai_csv: {asset.bonsai_event_log_csv}")
print(f"qc_dir: {asset.qc_dir}")
print(f"derived_dir: {asset.derived_dir}")

In [ ]:
fig, ax = display_meanim(
    exp,
    dmd=DMD_FOR_MEANIM,
    image_type=MEANIM_IMAGE_TYPE,
    channel=MEANIM_CHANNEL,
)

In [ ]:
cfg = MovieRenderConfig(
    input_frame_rate_hz=80.0,
    output_frame_rate_hz=20.0,
    downsample_factor_time=0,
    baseline_window_s=0.35,
    baseline_mode="fast",
    channel_index=0,
    activity_percentile=99.99,
    structure_percentile=99.95,
    gamma=0.85,
    transpose_xy=True,
    show_timer=True,
    show_scale_bar=True,
    pixel_size_um=0.25,
    scale_bar_um=20.0,
    activity_gaussian_sigma_px=0.8,
    structure_gaussian_sigma_px=0.6,
    activity_temporal_sigma_frames=0.75,
    upsample_factor=4.0,
    upsample_mode="bicubic",
    video_crf=12,
    video_preset="slow",
)

In [ ]:
asset.session_dir

In [ ]:
trial = 5

dmd1_path = glob.glob(os.path.join(asset.session_dir,'**',
                                   f'E1T{trial}DMD1_REGISTERED_DOWNSAMPLED-80Hz.tif'),
                      recursive=True)[0]

dmd2_path = glob.glob(os.path.join(asset.session_dir,'**',
                                   f'E1T{trial}DMD2_REGISTERED_DOWNSAMPLED-80Hz.tif'),
                      recursive=True)[0]

In [ ]:
inspect_tiff_layout(dmd1_path)

In [ ]:
dmd = 2

output_path = os.path.join(asset.session_dir,f"dmd{dmd}_trial{trial}_inset.mp4")

In [ ]:
render_glutamate_df_movie(
    output_path=output_path,
    config=cfg,
    dmd1_tiff=dmd1_path,
    dmd2_tiff=dmd2_path,
    dmd_selection=f"dmd{dmd}",
#         crop_bbox=(331, 644, 70, 480),
    n_channels=1,
    start_s=0,
    end_s=9,
    overwrite=True,
    verbose=True,
)